# RAPID response prediction: expanded grouped-CV grid

This notebook screens geometric predictors and regression models for two separate RAPID responses: timing shift and peak-discharge change. It uses leave-one-example-out validation throughout: no states from an example are used to predict held-out states from that same example.

The result is an exploratory screening result, not final unbiased performance: selected candidates should be checked against new examples or with nested grouped CV.

## Candidate predictors

Resistance / footprint: `len_fraction_width_weighted_log_lambda`, `length_fraction`, `width_fraction`, `k_ratio_to_matched_base`, `along_network_frac`.

Geometry / topology: `equivalent_length`, `elongation`, `topologic_complexity_score`, `effective_n_paths_width`, `dominant_width_fraction`, `path_length_cv`, `branching_density_by_length`.

For an independent collapse the topology variables are taken from its base-state unit. For a compound bubble, they are the equivalent-width-weighted mean of its member units' base-state values. The run writes this provenance to `topology_metric_provenance.csv`.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().resolve()
OUTPUT_ROOT = Path('/Volumes/PhD/river_hierarchy/network_variants/output')
EXAMPLE_IDS = ['02', '03', '05', '11', '13', '15', '21', '56']
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = REPO_ROOT / 'RAPID/classification/results' / f'rapid-response-grid-v1-{STAMP}'

assert (REPO_ROOT / 'RAPID/classification/search_rapid_response_predictors.py').exists()
assert OUTPUT_ROOT.exists(), f'Missing RAPID output root: {OUTPUT_ROOT}'
RUN_DIR

AssertionError: 

## Run the search

The grid evaluates 299 predictor sets (all one-, two-, and three-variable subsets plus the full 12-variable set), 12 model settings, 8 leave-one-example-out folds, and two targets. It may take a substantial time; results are written incrementally only after the full search completes, so leave this cell running.

In [ ]:
command = [
    sys.executable,
    str(REPO_ROOT / 'RAPID/classification/search_rapid_response_predictors.py'),
    '--output-root', str(OUTPUT_ROOT),
    '--results-dir', str(RUN_DIR),
    '--example-ids', *EXAMPLE_IDS,
]
completed = subprocess.run(command, cwd=REPO_ROOT, check=True, text=True, capture_output=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)

## Review grouped-CV screening results

Lower MAE is better. Do not interpret the first row as a final winner: it is the best configuration *within this screening grid*, and it is optimistic because the same folds were used to select it. Prefer a similarly accurate but simpler and physically interpretable predictor set.

In [ ]:
scores = pd.read_csv(RUN_DIR / 'grouped_cv_grid_scores.csv')
scores = scores.sort_values(['target', 'mae_pct_points', 'n_features', 'rmse_pct_points'])
for target, table in scores.groupby('target', sort=False):
    print(f'\n{target}')
    display(table.head(15).reset_index(drop=True))

In [ ]:
cohort = pd.read_csv(RUN_DIR / 'common_complete_case_cohort.csv')
provenance = pd.read_csv(RUN_DIR / 'topology_metric_provenance.csv')
print(f'Complete-case states: {len(cohort)} across {cohort.example_id.nunique()} examples')
display(cohort.groupby(['example_id', 'mode']).size().rename('states').reset_index())
display(provenance)

## Observed versus held-out prediction

Each point below is a prediction from the fold that held out its entire example. The 1:1 line is perfect prediction. This plot is useful for seeing systematic bias, outliers, and whether the apparent score is driven by a few examples.

In [ ]:
predictions = pd.read_csv(RUN_DIR / 'grouped_cv_grid_predictions.csv')
best = scores.groupby('target', as_index=False).first()
fig, axes = plt.subplots(1, len(best), figsize=(7 * len(best), 5), squeeze=False)
for axis, row in zip(axes.ravel(), best.itertuples(index=False)):
    plot_data = predictions[(predictions.target == row.target) & (predictions.model == row.model) & (predictions.features == row.features)]
    axis.scatter(plot_data.observed, plot_data.predicted, alpha=0.75)
    limits = [min(plot_data.observed.min(), plot_data.predicted.min()), max(plot_data.observed.max(), plot_data.predicted.max())]
    axis.plot(limits, limits, '--', color='black', linewidth=1)
    axis.set(xlabel='Observed response (% change)', ylabel='Held-out prediction (% change)', title=f'{row.target}\nMAE {row.mae_pct_points:.3f}; {row.model}')
plt.tight_layout()